# Audio lab: see, paint, and hear a sound

This optional notebook lets you inspect one anonymous recording and make a new sound by painting a time-frequency magnitude image. It uses NumPy, SciPy, Matplotlib, and IPython display only. Run it from the `student_release` folder. The painted image does not contain phase, so reconstruction is approximate.


In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
from IPython.display import Audio, display


## 1. Inspect a real example

The top panel is the waveform (amplitude over time). The bottom panel is an STFT magnitude image (frequency over time). Change `example_index` and rerun to explore another clip.


In [ ]:
with Path("dataset/manifest.csv").open(newline="") as f:
    manifest = list(csv.DictReader(f))
example_index = 0
row = manifest[example_index]
sr, pcm = wavfile.read(Path("dataset/audio") / row["id"])
x = pcm.astype(np.float32) / 32768.0
assert sr == 16000 and x.ndim == 1
frequency, time, spectrum = signal.stft(x, fs=sr, nperseg=512, noverlap=384)
db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-5))
fig, axes = plt.subplots(2, 1, figsize=(11, 6), constrained_layout=True)
axes[0].plot(np.arange(len(x)) / sr, x, lw=0.5)
axes[0].set(xlabel="Time (s)", ylabel="Amplitude", title=f"Waveform: {row['label']}")
mesh = axes[1].pcolormesh(time, frequency, db, shading="auto", cmap="magma")
axes[1].set(xlabel="Time (s)", ylabel="Frequency (Hz)", ylim=(0, 4000), title="STFT log magnitude")
fig.colorbar(mesh, ax=axes[1], label="dB")
plt.show()
display(Audio(x, rate=sr))


## 2. Paint a sound

The matrix below is a grayscale time-frequency image. Each call to `paint` fills a rectangle: time interval, frequency interval, and brightness (magnitude). Edit the calls to make your own pattern. For instance, stack horizontal bands to add harmonics, or change frequency from one time interval to another.


In [ ]:
sr_draw = 8000
duration = 1.0
n_fft = 256
hop = 64
frequency, time, _ = signal.stft(
    np.zeros(int(sr_draw * duration)), fs=sr_draw, nperseg=n_fft,
    noverlap=n_fft - hop, boundary="zeros", padded=True
)
canvas = np.zeros((len(frequency), len(time)), dtype=np.float64)
def paint(t0, t1, f0, f1, brightness=0.18):
    """Paint one time-frequency rectangle on the magnitude canvas."""
    mask = (frequency[:, None] >= f0) & (frequency[:, None] <= f1) & (time[None, :] >= t0) & (time[None, :] < t1)
    canvas[mask] = brightness

paint(0.08, 0.35, 420, 500)
paint(0.08, 0.35, 850, 950, 0.10)
paint(0.40, 0.75, 650, 750)
paint(0.40, 0.75, 1300, 1400, 0.08)

plt.figure(figsize=(11, 3))
plt.pcolormesh(time, frequency, canvas, shading="auto", cmap="magma")
plt.ylim(0, 2500)
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Your painted magnitude image")
plt.colorbar(label="Magnitude")
plt.show()


## 3. Convert the image to audio

A magnitude image has no phase. Griffin–Lim alternates between an inverse STFT and STFT to estimate a consistent phase. Listen, then edit your painting above and compare. The result may sound buzzy because the image is coarse and phase is estimated.


In [ ]:
rng = np.random.default_rng(745)
phase = np.exp(1j * rng.uniform(-np.pi, np.pi, size=canvas.shape))
for _ in range(32):
    _, audio = signal.istft(
        canvas * phase, fs=sr_draw, nperseg=n_fft,
        noverlap=n_fft - hop, input_onesided=True, boundary=True
    )
    _, _, estimate = signal.stft(
        audio, fs=sr_draw, nperseg=n_fft,
        noverlap=n_fft - hop, boundary="zeros", padded=True
    )
    estimate = estimate[:, :canvas.shape[1]]
    phase = estimate / np.maximum(np.abs(estimate), 1e-12)
_, painted_audio = signal.istft(
    canvas * phase, fs=sr_draw, nperseg=n_fft,
    noverlap=n_fft - hop, input_onesided=True, boundary=True
)
painted_audio = painted_audio[:int(sr_draw * duration)]
assert np.isfinite(painted_audio).all() and np.max(np.abs(painted_audio)) > 0
display(Audio(painted_audio, rate=sr_draw))


### Try this

1. Move the second rectangle to a higher frequency. How does the pitch change?
2. Add a vertical stripe across many frequencies. What kind of sound does it make?
3. Rerun phase reconstruction with a different random seed. Does the image specify one unique waveform?
4. Why might a 2D CNN find the painted horizontal bands more easily than a very small 1D CNN?
